In [35]:
import os
import re
import json
import pickle
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split
import pyarabic.araby as araby

In [36]:
PAD, UNK, SOS, EOS = 0, 1, 2, 3

In [37]:
#1. load & merge datasets

# 1. Kaggle
df1 = pd.read_csv("data.csv", encoding="utf-8")
df1 = df1.rename(columns={"Processed Text": "article", "summarizer": "summary"})
df1 = df1[["article", "summary"]].dropna()
df1

,article,summary
0,اشرف رئيس الجمهوريه الباجي قايد السبسي اليوم ب...,\nأشرف رئيس الجمهورية الباجي قايد السبسي اليوم...
1,تحصل كتاب المصحف وقراءاته الفه باحثون تونسيون ...,"\nتحصل كتاب ""المصحف وقراءاته"" الذي ألفه باحثون..."
2,احتضن جناح تونس القريه الدوليه للافلام بمدينه ...,تونس حاضرة من جهة أخرى ستكون تونس حاضرة في قائ...
3,تنطلق فعاليات التظاهره الموسيقيه الالكترونيه ص...,\nتنطلق فعاليات التظاهرة الموسيقية الالكترونية...
4,ينطلق مهرجان القيروان للشعر العربي ببيت الشعر ...,وستلتأم خلال اليوم الثاني الندوة النقدية بعنوا...
...,...,...
5222,تمكنت وحدات الحرس البحري بقليبيه احباط عمليه ه...,و قد تم إيقاف 13 شخصا تتراوح أعمارهم بين 17 و ...
5223,تعلم الشركه التونسيه للكهرباء والغاز اقليم سوس...,مدينة مساكن : الكنايس – سيدي الهاني – الفرادة ...
5224,كشف الناشطان كريم نوار وعفيف زقيه اشرفا عمليه ...,وأكدا ان تكلفة العلم لم تتجاوز 3 آلاف دينار مق...
5225,فرقه الابحاث والتفتيش للحرس الوطني بطبلبه ولاي...,كما تمّ إلقاء القبض على عنصر رابع (عمره 32 سنة...


In [38]:
# 2. Summset
df2 = pd.read_csv("summset.csv", encoding="utf-8")
df2 = df2.rename(columns={"text": "article", "summarizer": "summary"})
df2 = df2[["article", "summary"]].dropna()

df2

,article,summary
0,\nأشرف رئيس الجمهورية الباجي قايد السبسي اليوم...,\nأشرف رئيس الجمهورية الباجي قايد السبسي اليوم...
1,"\nتحصل كتاب ""المصحف وقراءاته"" الذي ألفه باحثون...","\nتحصل كتاب ""المصحف وقراءاته"" الذي ألفه باحثون..."
2,\nاحتضن جناح تونس في القرية الدولية للأفلام بم...,تونس حاضرة من جهة أخرى ستكون تونس حاضرة في قائ...
3,\nشهدت برلين أمس الجمعة افتتاح مسجد فريد من نو...,واستأجرت صاحبة المشروع المحامية والكاتبة سيران...
4,\nنعت وزارة الشّؤون الثّقافيّة المنشد الصّوفي ...,\nنعت وزارة الشّؤون الثّقافيّة المنشد الصّوفي ...
...,...,...
8373,\nتأجل الإضراب العام في قطاع الصحة الذي كان مق...,وكانت انعقدت عشية أمس في وزارة الشؤون الاجتماع...
8374,\nكشف الناشطان كريم نوار وعفيف زقية اللذان أشر...,وأكدا ان تكلفة العلم لم تتجاوز 3 آلاف دينار مق...
8375,\nتمكّنت فرقة الأبحاث والتفتيش للحرس الوطني بط...,كما تمّ إلقاء القبض على عنصر رابع (عمره 32 سنة...
8376,\nقرر الأهالي بمناطق هيشر وعين القارصي والغولا...,وتأتي هذه الخطوة على خلفية ما اعتبره أهالي هذه...


In [39]:
# 3. SumArabic (3 JSONL files)
records = []
for path in ["sumarabic-1.0-train.jsonl", "sumarabic-1.0-valid.jsonl", "sumarabic-1.0-test.jsonl"]:
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            if obj.get("text") and obj.get("headline"):
                records.append({"article": obj["text"], "summary": obj["headline"]})
df3 = pd.DataFrame(records)
df3

,article,summary
0,اختتمت مساء أول من أمس نهائيات بطولة الإمارات ...,المصري فؤاد الطاهر بطل للشطرنج الديناميكي
1,مينيابوليس (الولايات المتحدة) 3-2-2008 (ا ف ب)...,"حملة انتخابية محمومة قبل ""الثلاثاء الكبير"""
2,أفاد مصدر في شرطة أبوظبي بأن نحو 700 طفل يتواف...,55% من نزلاء «الأحداث» مواطنون
3,قال استشاري أمراض الكلى في مدينة الشيخ خليفة ا...,مركز متخصص في الكلى بمستشفى خليفة
4,بدأت القيادة العامة لشرطة أبوظبي التنسيق لإنشا...,تأهيل ضحايا حوادث المرور في أبوظبي
...,...,...
2623,اصدرت محكمة في المكسيك حكما بالسجن مدى الحياة ...,السجن مدى الحياة لمصارعة سابقة قتلت 11 امرأة م...
2624,أصدرت المحكمة الجنائية العراقية العليا اليوم ح...,"حكم بالإعدام على ""الكيمياوي"" في قضية قمع ""الان..."
2625,فسّر الكاتب المصري الأسير السابق لدى الجيش الإ...,رواية عن التكوين النفسي للجيش الإسرائيلي
2626,متواصلت، أمس، عمليات البيع في أسواق المال المح...,تراجع الأسهم مع إغلاق مستثمرين مراكزهم المكشوفة


In [40]:
# 4. AbsSum (no header — col 1 = article, col 2 = summary)
# 4. AbsSum — tab separated, no header: col0=index, col1=article, col2=summary
df4 = pd.read_csv("AbsSum.csv", encoding="utf-8", sep="\t",
                  header=None, on_bad_lines="skip", engine="python")
df4 = df4[[1, 2]].copy()
df4.columns = ["article", "summary"]
df4 = df4.dropna()

df4


,article,summary
0,"حقق حزب ""البديل من أجل ألمانيا"" اليميني الشعبو...",تشير آخر استطلاعات الرأي الألمانية إلى تقدم حز...
1,بدأت اليوم الجمعة( 23 أيلول/ سبتمبر 2016 ) في ...,بدأت محكمة ميونخ النظر في اتهامات متعلقة برجل ...
2,قال مسؤولون إن شخصا أصيب إصابة بالغة بعد تعرضه...,أعلن حاكم كارولاينا الشمالية حالة الطورائ في م...
3,أعلن مسؤول في البنتاغون أن جنودا أمريكيين في ش...,قالت وزارة الدفاع الأمريكية (البنتاغون) إن داع...
4,اجتمعت المجموعة الدولية لدعم سوريا على هامش اج...,فشلت الولايات المتحدة وروسيا في الاتفاق على كي...
...,...,...
49599,لا يعني استخدام الكثير من مستحضرات التجميل الم...,يؤكد الكثير من النساء على فاعلية بعض مستحضرات ...
49600,بدأت الاحتجاجات في سوريا سلمية يوم 15 مارس/ آذ...,أكمل النزاع في سوريا عامه السابع دون أن يتوقف ...
49601,لم تعد فصول السنة، حتى في ألمانيا، كما كانت عل...,كشفت دراسة حديثة للهيئة الاتحادية لحماية الطبي...
49602,"تبعات حلقة برنامج ""شباب توك"" التي بُثت من العا...","مطالبة شابة سودانية خلال حلقة برنامج ""شباب توك..."


In [41]:
# 5. Egyptian
df5 = pd.read_parquet("Egy.parquet")
df5 = df5.rename(columns={"text": "article", "summarized_text": "summary"})
df5 = df5[["article", "summary"]].dropna()

df5

,article,summary
0,كتير من الشباب دلوقتي بيفكروا ألف مرة قبل ما ي...,غلاء المعيشة بيخلي الشباب يقللوا عدد الأطفال ب...
1,الظروف الاقتصادية الصعبة خلت كتير من الشباب يأ...,الوضع الاقتصادي المتردي بيأجل قرارات الجواز وا...
2,موجات التضخم المتتالية أثرت بشكل مباشر على قدر...,التضخم بيخلي الأسر تعيد حساباتها في قرارات الإ...
3,الأسر المصرية بقت بتميل أكتر لإنها تركز على جو...,الأعباء الاقتصادية بتخلي الأسر تركز على جودة ح...
4,في ظل التحديات الاقتصادية اللي بتواجه الأسر، ب...,الحكومات ممكن تحتاج تقدم دعم وحوافز للأسر عشان...
...,...,...
3684,في ظل التطور التكنولوجي الحالي، أصبح الاعتماد ...,الفيديوهات التعليمية بقت ضرورية للتعليم عن بعد...
3685,مش بس تسجيل المحاضرات، استخدام الفيديو في التع...,الفيديوهات التفاعلية بتزود انخراط الطلاب في ال...
3686,في مجالات كتير زي العلوم والهندسة والطب، بيكون...,الفيديوهات مهمة جدًا لتقديم التجارب العملية وا...
3687,استخدام الفيديو مش لازم يقتصر على الأستاذ بس. ...,تكليف الطلاب بعمل فيديوهات كمشاريع دراسية بيشج...


In [42]:
# Merge all + remove duplicates
df = pd.concat([df1, df2, df3, df4, df5], ignore_index=True)
df = df.drop_duplicates(subset=["article", "summary"]).reset_index(drop=True)
print(f"Total rows after merge: {len(df)}")

df

Total rows after merge: 68984


,article,summary
0,اشرف رئيس الجمهوريه الباجي قايد السبسي اليوم ب...,\nأشرف رئيس الجمهورية الباجي قايد السبسي اليوم...
1,تحصل كتاب المصحف وقراءاته الفه باحثون تونسيون ...,"\nتحصل كتاب ""المصحف وقراءاته"" الذي ألفه باحثون..."
2,احتضن جناح تونس القريه الدوليه للافلام بمدينه ...,تونس حاضرة من جهة أخرى ستكون تونس حاضرة في قائ...
3,تنطلق فعاليات التظاهره الموسيقيه الالكترونيه ص...,\nتنطلق فعاليات التظاهرة الموسيقية الالكترونية...
4,ينطلق مهرجان القيروان للشعر العربي ببيت الشعر ...,وستلتأم خلال اليوم الثاني الندوة النقدية بعنوا...
...,...,...
68979,في ظل التطور التكنولوجي الحالي، أصبح الاعتماد ...,الفيديوهات التعليمية بقت ضرورية للتعليم عن بعد...
68980,مش بس تسجيل المحاضرات، استخدام الفيديو في التع...,الفيديوهات التفاعلية بتزود انخراط الطلاب في ال...
68981,في مجالات كتير زي العلوم والهندسة والطب، بيكون...,الفيديوهات مهمة جدًا لتقديم التجارب العملية وا...
68982,استخدام الفيديو مش لازم يقتصر على الأستاذ بس. ...,تكليف الطلاب بعمل فيديوهات كمشاريع دراسية بيشج...


In [28]:
#2. clean and normalize

def preprocess(text):
    if not isinstance(text, str):
        return ""
    text = text.strip()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[\d٠-٩]", " ", text)
    text = re.sub(r"[،؛؟!\"#$%&'()*+,\-./:;<=>?@\[\]^_`{|}~]", " ", text)
    text = araby.strip_tashkeel(text)
    text = araby.normalize_alef(text)
    text = re.sub(r"ى", "ي", text)
    text = araby.normalize_teh(text)
    text = araby.strip_tatweel(text)
    text = re.sub(r"[^\u0600-\u06FF\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["article"] = df["article"].apply(preprocess)
df["summary"] = df["summary"].apply(preprocess)

In [29]:
# Drop rows that became empty after cleaning
df = df[(df["article"].str.len() > 10) & (df["summary"].str.len() > 5)]
df = df.reset_index(drop=True)
print(f"Total rows after cleaning: {len(df)}")

Total rows after cleaning: 68976


In [30]:
#3. Tokenize
df["article_tokens"] = df["article"].apply(str.split)
df["summary_tokens"]  = df["summary"].apply(str.split)

#4. build vocabulary
MAX_VOCAB = 40000
MIN_FREQ  = 2

In [31]:
counter = Counter()
for tokens in df["article_tokens"].tolist() + df["summary_tokens"].tolist():
    counter.update(tokens)

word2idx = {"<PAD>": 0, "<UNK>": 1, "<SOS>": 2, "<EOS>": 3}
for word, freq in counter.most_common():
    if freq < MIN_FREQ or len(word2idx) >= MAX_VOCAB:
        break
    word2idx[word] = len(word2idx)

idx2word = {i: w for w, i in word2idx.items()}
print(f"\nVocabulary size: {len(word2idx)}")


Vocabulary size: 40000


In [32]:
#5. encode & pad
MAX_ARTICLE = 200
MAX_SUMMARY = 80

def encode(tokens, max_len):
    ids = [SOS] + [word2idx.get(t, UNK) for t in tokens] + [EOS]
    ids = ids[:max_len]
    ids += [PAD] * (max_len - len(ids))
    return ids

X = np.array([encode(t, MAX_ARTICLE) for t in df["article_tokens"]], dtype=np.int32)
y = np.array([encode(t, MAX_SUMMARY)  for t in df["summary_tokens"]],  dtype=np.int32)
print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")


X shape: (68976, 200)
y shape: (68976, 80)


In [33]:
#6. train, validate, test split 80/10/10

X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.2, random_state=42)
X_val,  X_test, y_val,  y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=42)

print(f"\nTrain: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")


Train: (55180, 200), Val: (6898, 200), Test: (6898, 200)


In [34]:
#7. Save
os.makedirs("preprocessed", exist_ok=True)

np.save("preprocessed/X_train.npy", X_train)
np.save("preprocessed/y_train.npy", y_train)
np.save("preprocessed/X_val.npy",   X_val)
np.save("preprocessed/y_val.npy",   y_val)
np.save("preprocessed/X_test.npy",  X_test)
np.save("preprocessed/y_test.npy",  y_test)

with open("preprocessed/word2idx.pkl", "wb") as f: pickle.dump(word2idx, f)
with open("preprocessed/idx2word.pkl", "wb") as f: pickle.dump(idx2word, f)
with open("preprocessed/vocab.json", "w", encoding="utf-8") as f:
    json.dump(word2idx, f, ensure_ascii=False, indent=2)

print("Done! Files saved to preprocessed")

Done! Files saved to preprocessed
